# CML DQM Implementation Starter Notebook

This notebook provides a runnable starter workflow for ML4DQM using the new `tools.dqm` components.

## 1. Environment Setup and Dependency Installation

In [1]:
import importlib
import platform
import sys

print(f"Python: {platform.python_version()}")
for pkg in ["numpy", "matplotlib", "torch", "orchestral"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg}: {getattr(m, '__version__', 'unknown')}")
    except Exception as e:
        print(f"{pkg}: not available ({e})")

Python: 3.13.9
numpy: 2.4.3
matplotlib: 3.10.8
torch: 2.9.0
orchestral: unknown


## 2. Import Modules and Define Global Configuration

In [4]:
import json
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "tools").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "tools").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "tools").exists():
    raise RuntimeError("Could not locate repository root containing tools/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.dqm import (
    DQMEDATool,
    DQMTrainTool,
    DQMEvaluateTool,
    DQMDeploymentTool,
)

WORKDIR = REPO_ROOT
CFG = {
    "run_id": 323997,
    "base_dir": "examples/workflows/cml_dqm_sandbox/sandbox000",
    "eda_output_dir": "examples/workflows/cml_dqm_sandbox/sandbox000/eda",
    "epochs": 2,
    "batch_size": 64,
    "learning_rate": 3e-3,
    "eval_batch_size": 32,
}

print("Repo root:", REPO_ROOT)
print("Sandbox base:", CFG["base_dir"])

Repo root: /Users/dakshmor/Documents/GitHub/heptapod
Sandbox base: examples/workflows/cml_dqm_sandbox/sandbox000


## 3. Create Synthetic/Input Data for Development

In [5]:
run_id = CFG["run_id"]
train_path = REPO_ROOT / f"data/dataset/he_train_dataset_Run{run_id}/train_data.npy"
test_path = REPO_ROOT / f"data/dataset/he_test_dataset_Run{run_id}/test_data.npy"

assert train_path.exists(), f"Missing train dataset: {train_path}"
assert test_path.exists(), f"Missing test dataset: {test_path}"

train_data = np.load(train_path)
test_data = np.load(test_path)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (398, 64, 72, 7, 1)
Test shape: (100, 64, 72, 7, 1)


## 4. Implement Core Functions (Initial Version)

In [6]:
def run_eda(base_dir: Path, cfg: dict) -> dict:
    dataset_rel = f"data/dataset/he_train_dataset_Run{cfg['run_id']}/train_data.npy"
    tool = DQMEDATool(
        base_directory=str(base_dir),
        dataset_path=dataset_rel,
        output_dir=cfg["eda_output_dir"],
    )
    return json.loads(tool._run())


def run_train(base_dir: Path, cfg: dict) -> dict:
    tool = DQMTrainTool(
        RUNS=[cfg["run_id"]],
        base_dir=cfg["base_dir"],
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        learning_rate=cfg["learning_rate"],
    )
    return json.loads(tool._run())


def run_eval(base_dir: Path, cfg: dict) -> dict:
    tool = DQMEvaluateTool(
        runs=[cfg["run_id"]],
        base_dir=cfg["base_dir"],
        batch_size=cfg["eval_batch_size"],
    )
    return json.loads(tool._run())

print("Core helper functions ready")

Core helper functions ready


## 5. Run a Minimal End-to-End Pipeline

In [ ]:
eda_out = run_eda(WORKDIR, CFG)
train_out = run_train(WORKDIR, CFG)
eval_out = run_eval(WORKDIR, CFG)

pipeline_out = {
    "eda": eda_out,
    "train": train_out,
    "eval": eval_out,
}

print(json.dumps(pipeline_out, indent=2))

## 6. Add Assertions and Quick Validation Checks

In [ ]:
assert eda_out["status"] == "ok"
assert train_out["status"] == "ok"
assert eval_out["status"] == "ok"

model_path = Path(train_out["model_path"])
if not model_path.is_absolute():
    model_path = REPO_ROOT / model_path
assert model_path.exists()

plots_path = REPO_ROOT / eval_out["plots_dir"]
assert plots_path.exists()

print("All quick validation checks passed")

## 7. Refactor into Reusable Helper Blocks

In [ ]:
def run_full_pipeline(base_dir: Path, cfg: dict) -> dict:
    eda = run_eda(base_dir, cfg)
    train = run_train(base_dir, cfg)
    evaluate = run_eval(base_dir, cfg)

    model_path = train["model_path"]
    if Path(model_path).is_absolute():
        model_rel = str(Path(model_path).resolve().relative_to(base_dir.resolve()))
    else:
        model_rel = model_path

    deploy_tool = DQMDeploymentTool(
        base_directory=str(base_dir),
        model_path=model_rel,
        endpoint_name="he-dqm-depthvit-v1",
        runtime_target="cms-dqm-stream",
    )
    deploy = json.loads(deploy_tool._run())

    return {"eda": eda, "train": train, "evaluate": evaluate, "deploy": deploy}


rerun = run_full_pipeline(WORKDIR, CFG)
print(json.dumps(rerun["deploy"], indent=2))

## Deployment Note

`DQMDeploymentTool` is a planning scaffold for now. It validates artifact paths and returns a structured deployment handoff checklist for future real-time CMS DQM integration.